# Capstone — Content Refresh Prioritization

This capstone brings the Week 1–7 work together into a public-safe research artifact. The central decision is **which content pages should receive human review for possible refresh first**.

> The analysis uses observable search/content performance signals and an interpretable model. It is decision support, not proof that a refresh will cause recovery.

## 1. Question

### Research question
**Can observable content and search-performance signals help prioritize pages that may be declining, so a content team can decide which pages deserve human review first?**

### Decision supported
The output is a ranked review queue. A high-ranked page is a candidate for human investigation—not an automatic instruction to rewrite, publish, delete, or redirect a page.

### Why this matters
A content team cannot investigate every page equally. A transparent ranking can combine signals such as staleness and search performance so review effort is focused where the measured risk is higher.

In [ ]:
print("Research question: prioritize potentially declining content for human review.")
print("Decision: rank pages for review/possible refresh.")
print("Guardrail: model output is decision-support, not an automatic content action.")

## 2. Data

### Data sources and scope
The internship warehouse release contains a daily content-performance fact table with **78,835,655 rows**, at daily × client × content grain, plus content and query dimensions. The public FlyRank research snapshot covers **341,701 content pieces across 57 brands**. These figures describe the underlying production-style search/content data while the ML evaluation below uses the provided anonymized lane dataset.

### ML evaluation dataset
- Starter dataset: **30,000 rows × 44 columns**
- Unit of analysis: one anonymized content item/page
- Proxy label: `is_declining_label = 1` when `trend_direction == "down"`
- `trend_direction` and `trend_pct` are excluded from model features because they encode the outcome.

### Time and windowing
The warehouse work used explicit feature/label windows and client-level history checks. The underlying daily data is an unbalanced panel: different clients have different history coverage. Missing tracking is therefore not interpreted as zero traffic.

### Public-safe exclusions
Client names, domains, raw queries, credentials, and private warehouse details are excluded from the public artifact. IDs are used only as grouping/join context and are not predictive features.

In [ ]:
import pandas as pd

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Expected starter shape: 30000 rows x 44 columns")

## 3. Methodology

### Label definition
For the ML exercise, `down` is treated as the positive class:

`is_declining_label = 1 if trend_direction == "down", otherwise 0`.

This is a **proxy label**, not a direct measurement of future content success.

### Features
The model uses observable content/performance fields available in the starter dataset. Identifiers and outcome-derived fields are excluded, including `content_id`, `client_id`, `trend_direction`, `trend_pct`, `is_declining_label`, and product/output fields such as `health_score`, `priority_score`, `action_type`, and `recommended_action` when present.

### Baseline
The Week-4 baseline is a transparent rule-based score combining content staleness and CTR weakness relative to search position.

### Model
An interpretable **Decision Tree** was selected because it can learn non-linear combinations of observable signals while remaining easy to inspect.

### Validation
The model was evaluated with a **client-grouped holdout**: clients in the test set do not appear in the training set. This avoids the optimistic evaluation that can occur when related pages from the same client appear on both sides of the split.

### Leakage checks
The audit checks that target-derived fields are not model features and that `client_id` is used only for grouping, not prediction.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

exclude = {
    "content_id", "client_id", "trend_direction", "trend_pct",
    "is_declining_label", "health_score", "priority_score",
    "action_type", "recommended_action"
}

feature_cols = [
    c for c in df.columns
    if c not in exclude and pd.api.types.is_numeric_dtype(df[c])
]

X = df[feature_cols].copy()
y = df["is_declining_label"].copy()

X = X.replace([float("inf"), float("-inf")], pd.NA)
X = X.fillna(X.median(numeric_only=True)).fillna(0)

clients = df["client_id"].dropna().unique()
train_clients, test_clients = train_test_split(
    clients, test_size=0.20, random_state=42
)

train_mask = df["client_id"].isin(train_clients)
test_mask = df["client_id"].isin(test_clients)

X_train, X_test = X.loc[train_mask], X.loc[test_mask]
y_train, y_test = y.loc[train_mask], y.loc[test_mask]

model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(X_train, y_train)

model_prob = model.predict_proba(X_test)[:, 1]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", df.loc[train_mask, "client_id"].nunique())
print("Test clients:", df.loc[test_mask, "client_id"].nunique())
print("Client overlap:", len(set(train_clients) & set(test_clients)))
print("Decision Tree ROC AUC:", round(roc_auc_score(y_test, model_prob), 4))
print("Decision Tree Average Precision:", round(average_precision_score(y_test, model_prob), 4))

### Leakage audit

The explicit audit is intentionally conservative. Outcome fields and product decision outputs are not predictive features, and client IDs are used only to create the grouped split.

**Expected audit result:** `trend_direction used as feature: False`, `trend_pct used as feature: False`, `target used as feature: False`, and `client_id used as feature: False`.

In [ ]:
audit_cols = {
    "trend_direction used as feature": "trend_direction" in feature_cols,
    "trend_pct used as feature": "trend_pct" in feature_cols,
    "target used as feature": "is_declining_label" in feature_cols,
    "client_id used as feature": "client_id" in feature_cols,
}

for name, value in audit_cols.items():
    print(f"{name}: {value}")

## 4. Results (vs baseline)

The same held-out client split was used for the model and the Week-4 baseline.

| Method | ROC AUC | Average Precision |
|---|---:|---:|
| Week-4 baseline | 0.5104 | 0.5207 |
| Week-5 Decision Tree | **0.6392** | **0.6105** |

The Decision Tree showed stronger predictive separation than the transparent baseline on this evaluated split. The result supports the narrower claim that the learned model captured useful signal for the declining-content proxy; it does **not** establish a causal effect on traffic or prove that refreshing a page will improve performance.

In [ ]:
import matplotlib.pyplot as plt

results = pd.DataFrame({
    "method": ["Week-4 baseline", "Week-5 Decision Tree"],
    "roc_auc": [0.510426, 0.639220],
    "average_precision": [0.520683, 0.610493],
})

display(results.round(4))

ax = results.set_index("method")[["roc_auc", "average_precision"]].plot(kind="bar")
ax.set_ylabel("Score")
ax.set_title("Model vs Week-4 baseline on the held-out client split")
ax.set_ylim(0, 0.75)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Result interpretation

The model improved ROC AUC by about **0.129** and Average Precision by about **0.090** relative to the Week-4 baseline on the recorded holdout evaluation.

This is a predictive result for the proxy label. It should not be interpreted as evidence that a content refresh causes recovery.

## 5. Limitations

1. **Proxy target:** `down` is a label derived from observed trend direction; it is not a direct business outcome.
2. **Observational data:** associations do not prove that staleness, CTR, or another signal causes decline.
3. **Client-grouped evaluation:** the split is more honest for cross-client generalization, but it does not establish performance for every future client.
4. **Feature scope:** the public starter evaluation is narrower than the full warehouse and does not reproduce every production pipeline feature.
5. **No causal refresh test:** this work does not randomly assign refreshes or otherwise identify the causal effect of refreshing content.
6. **Ranking is not an automatic action:** a high score can be wrong because search intent, seasonality, recent changes, business context, or data quality may not be represented.
7. **Model scope:** the Decision Tree is useful for this exercise, but the measured result is not a guarantee of future performance.

In [ ]:
limitations = [
    "Proxy label rather than direct future-success outcome",
    "Observational data; no causal identification",
    "Client-grouped holdout is not a guarantee of future generalization",
    "Starter ML dataset is narrower than the full warehouse",
    "No randomized or causal refresh experiment",
    "Ranked actions require human review",
    "Model predictions are not guarantees",
]

print("Limitations documented:", len(limitations))
for item in limitations:
    print("-", item)

## 6. Ranked recommendations

### Action playbook

Use the ranked queue to focus human review on pages with stronger measured risk signals.

**Priority 1 — STALE_AND_LOW_CTR**  
Review pages that are both stale and weak on CTR relative to their search-position context.

**Priority 2 — STALE_CONTENT**  
Review older/stale pages that still have enough visibility to justify an update.

**Priority 3 — LOW_CTR_FOR_POSITION**  
Review pages whose CTR is weak relative to their ranking position.

**Priority 4 — GENERAL_REVIEW**  
Use remaining signals as a lower-priority review queue.

### Human review before action
For every high-ranked page, a reviewer should check search intent/relevance, the reason for the observed performance signal, recent changes/context, whether refresh is appropriate, and how the outcome will be measured.

### No-go actions
Do not automatically publish, rewrite, delete, or redirect content from the score alone. Do not change important factual/legal/business information without appropriate review. Do not treat the ranking as causal proof.

In [ ]:
import numpy as np

queue = df[[
    "content_id", "client_id", "days_since_last_update", "ctr", "avg_position"
]].copy()

queue["staleness_score"] = queue["days_since_last_update"].rank(pct=True).fillna(0)

position_bucket = pd.cut(
    queue["avg_position"],
    bins=[-np.inf, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"]
)

bucket_median = queue.groupby(position_bucket, observed=False)["ctr"].transform("median")
queue["ctr_gap"] = (bucket_median - queue["ctr"]).clip(lower=0)
queue["ctr_gap_score"] = queue["ctr_gap"].rank(pct=True).fillna(0)

queue["action_score"] = (
    0.60 * queue["staleness_score"] +
    0.40 * queue["ctr_gap_score"]
) * 100

queue["reason_code"] = "GENERAL_REVIEW"
stale = queue["days_since_last_update"] >= queue["days_since_last_update"].median()
low_ctr = queue["ctr_gap"] > 0

queue.loc[stale & low_ctr, "reason_code"] = "STALE_AND_LOW_CTR"
queue.loc[stale & ~low_ctr, "reason_code"] = "STALE_CONTENT"
queue.loc[~stale & low_ctr, "reason_code"] = "LOW_CTR_FOR_POSITION"

queue["action"] = "REVIEW"
queue.loc[queue["reason_code"].isin(
    ["STALE_AND_LOW_CTR", "STALE_CONTENT", "LOW_CTR_FOR_POSITION"]
) , "action"] = "REVIEW_REFRESH"

action_queue = queue.sort_values("action_score", ascending=False).reset_index(drop=True)
action_queue["rank"] = action_queue.index + 1

display(action_queue.head(10))
print("Ranked queue rows:", len(action_queue))

## 7. Artifacts the paper embeds

The public paper should show a small number of useful artifacts rather than a large dashboard:
- **Model-vs-baseline chart:** communicates whether the learned model adds predictive value.
- **Validation summary:** shows the client-grouped holdout and base-rate context.
- **Ranked action example:** demonstrates how a score becomes a human-review queue.
- **Methodology/limitations callout:** prevents readers from interpreting the model as causal proof.

In [ ]:
artifact_summary = pd.DataFrame({
    "artifact": [
        "Model vs baseline chart",
        "Client-grouped validation summary",
        "Ranked action queue",
        "Methodology + limitations callout"
    ],
    "purpose": [
        "Show predictive comparison",
        "Show evaluation honesty",
        "Show practical decision support",
        "Prevent causal over-interpretation"
    ]
})

display(artifact_summary)

print("Key recorded result:")
print("Decision Tree ROC AUC = 0.6392")
print("Decision Tree Average Precision = 0.6105")
print("Week-4 baseline ROC AUC = 0.5104")
print("Week-4 baseline Average Precision = 0.5207")

## 8. Reproducibility

The analysis notebooks are stored in `work/notebooks/` in the GitHub repository. The capstone notebook records the question, data framing, methodology, validation, results, limitations, recommendations, and artifacts.

Repository: https://github.com/ruchitgoud/flyrankai-intern

Relevant notebooks:
- Week 1 — research question
- Week 2 — ML task framing
- Week 3 — data contract
- Week 4 — baseline/action score
- Week 5 — model
- Week 6 — validation audit
- Week 7 — action playbook
- Capstone — consolidated paper analysis

## 9. Acknowledgments & Data Credit

This work was completed as part of the **FlyRank ML Internship** and uses the internship's anonymized search/content dataset and warehouse release.

Data credit: **FlyRank** — https://flyrank.ai

The public version intentionally excludes client names, domains, private queries, credentials, and other sensitive internal details.

## Week 8 — 5-Minute Demo Outline

### 1. Question — ~45 seconds
Which content pages should a content team prioritize for refresh or human review?

### 2. Method — ~1 minute
Use anonymized content-performance data, an interpretable Decision Tree, and a client-grouped holdout. Compare the model with a transparent Week-4 baseline.

### 3. One chart — ~1 minute
Show the model-vs-baseline ROC AUC / Average Precision chart.

### 4. Honest result — ~1 minute
The Decision Tree achieved ROC AUC **0.6392** and Average Precision **0.6105**, versus **0.5104** and **0.5207** for the Week-4 baseline on the evaluated holdout. This shows stronger predictive signal for the declining-content proxy; it does not prove that refreshing content causes recovery.

### 5. Recommendation — ~45 seconds
Use the ranked queue to focus human review. Check intent, context, recent changes, and measurement before changing content; do not automate publishing, rewriting, deletion, or redirection from the score alone.

## Shareable Cut — Social Post

I built a content-refresh prioritization workflow during my FlyRank ML internship.

I used anonymized search/content performance data, an interpretable Decision Tree, and a client-grouped validation split to compare ML against a transparent baseline.

The model showed stronger predictive signal for the declining-content proxy, and the final workflow turns that signal into a ranked human-review queue—while keeping predictive evidence separate from causal claims.

## Shareable Cut — Employer-Facing Summary

I built a machine-learning workflow to prioritize content pages for human review using anonymized search and content-performance data.

I trained and evaluated an interpretable Decision Tree with a client-grouped holdout and compared it against a transparent baseline, with ROC AUC of 0.6392 versus 0.5104 and Average Precision of 0.6105 versus 0.5207.

The final workflow produces ranked actions and reason codes for content review while treating the model as decision support rather than proof that a refresh will improve performance.

## Self-check

- [x] Every capstone section is filled with markdown thinking and supporting code.
- [x] The notebook includes the recorded model-vs-baseline results.
- [x] Client IDs are used for grouping, not as model features.
- [x] Target-derived fields are excluded from predictive features.
- [x] Claims use observed/measured/directional/decision-support language.
- [x] No client names, domains, private queries, or credentials are included.
- [x] Week 8 demo outline is included.
- [x] Social-post cut is included.
- [x] Employer-facing 3-sentence summary is included.
- [ ] Deployed paper URL recorded in `submission/paper_url.txt` after deployment.